In [ ]:
# | default_exp openai_cli

In [ ]:
# | export
from itertools import cycle
from functools import wraps
import fastcore.all as fc
from aiolimiter import AsyncLimiter
import backoff
from dataclasses import dataclass
from lite_utils.utils import add_decorator
try: from IPython.display import Markdown
except: pass

In [ ]:
# | export
class LLM:
    '''Base class for interacting with LLMs'''

    def __init__(self, model: str, sp: str = '', is_async=False, **kwargs):
        self.model, self.sp, self.is_async, self.kwargs = model, sp, is_async, kwargs

    def __call__(self, msgs_history: list, sp=None, **kwargs):
        '''make a call to LLM'''

    def configure(self, **kwargs):
        '''Configure parameters for LLM generation, e.g. temperature, top_p, etc.'''
        self.kwargs = kwargs

    @classmethod
    def mk_msg(cls, text: str, role: str = 'user', **kwargs):
        '''Prepare message in specific format for LLM provider'''

    @classmethod
    def mk_msgs(cls, msgs: list[str], roles: tuple = ('user', 'assistant'), **kwargs) -> list:
        '''Make list of messages in specific format for LLM provider.
        By default will call `self.mk_msg` for each message with alternating roles in `roles`
        '''
        if isinstance(msgs, str): msgs = [msgs]
        return [cls.mk_msg(msg, role, **kwargs) for msg, role in zip(msgs, cycle(roles))]

    @classmethod
    def contents(cls, msg) -> str:
        '''returns string content of message'''

    @classmethod
    def h2str(cls, history, md=True) -> str:
        '''convert history of messages into a string'''
        if md and globals().get('Markdown', None): history = Markdown(history)
        return history

## OpenAI

In [ ]:
# | export
import cosette as cs
from openai import OpenAI, AsyncOpenAI, RateLimitError, APIConnectionError
from openai.types.chat import ChatCompletion, ChatCompletionMessage

In [ ]:
# | export


class LimitedClient(cs.Client):
    def __init__(self, model, cli=None, rpm=500, backoff_pow=2, backoff_max=20):
        self.limiter, self.is_async = AsyncLimiter(rpm), isinstance(cli, AsyncOpenAI)
        self.backoff = backoff.on_exception(
            backoff.expo, (RateLimitError, APIConnectionError), base=backoff_pow, max_value=backoff_max)
        super().__init__(model, cli)

    @wraps(cs.Client.__call__)
    def __call__(self, *args, **kwargs):
        async def _f(*args, **kwds):
            async with self.limiter:
                return await super(LimitedClient, self).__call__(*args, **kwds)
        f = _f if self.is_async else super().__call__
        return self.backoff(f)(*args, **kwargs)


@fc.delegates(LimitedClient)
class OpenaiLLM(LLM):
    def __init__(self, model='gpt-4o-mini', sp='', is_async=False, base_url=None, api_key=None, **kwargs):
        super().__init__(model, sp, is_async)
        kwgs = dict(base_url=base_url, api_key=api_key)
        cli = (AsyncOpenAI(**kwgs) if is_async else OpenAI(**kwgs))
        self.cli = LimitedClient(model, cli, **kwargs)

    @fc.delegates(cs.mk_msg)
    def mk_msg(cls, text, role='user', **kwargs): 
        return cs.mk_msg(text, role, **kwargs)
    @fc.delegates(cs.mk_msgs)
    def mk_msgs(cls,msgs, **kwargs): return cs.mk_msgs(msgs, **kwargs)
    def contents(cls,msg): 
        if isinstance(msg, ChatCompletionMessage): return msg.content
        return cs.contents(msg)
    def h2str(cls,history, md=True):
        s = ''
        for h in history:
            if isinstance(h, ChatCompletionMessage): h = dict(h)
            if not isinstance(h, dict): h = OpenaiLLM.mk_msg(h)
            cts = h['content'] or '<NO CONTENT>'
            s += f"**{h['role']}**:\n\n" + (cts if isinstance(cts, str) else cts[0]['text'])
            if 'tool_calls' in h and h['tool_calls']: 
                s += '\n\n**Tool calls**:\n- '+'\n- '.join(f'`{tc.function.name}(**{tc.function.arguments})`' for tc in h['tool_calls'])
            s += '\n\n'
        return super().h2str(s, md)
    
    @fc.delegates(cs.Client.__call__)
    def __call__(self, msgs_history: list, sp=None, **kwargs):
        if sp or self.sp: msgs_history = [self.mk_msg(sp or self.sp, 'system')] + msgs_history
        kwargs = {**self.kwargs, **kwargs}
        return self.cli(msgs_history, **kwargs)
    
add_decorator(OpenaiLLM, ['mk_msg', 'mk_msgs', 'contents', 'h2str'], classmethod)

In [ ]:
llm = OpenaiLLM()

In [ ]:
llm(llm.mk_msgs('hello'))

Hello! How can I assist you today?

<details>

- id: chatcmpl-AW1ZuCAsyWfdFwQPwaOHG5Y6XdmU1
- choices: [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))]
- created: 1732195926
- model: gpt-4o-mini-2024-07-18
- object: chat.completion
- service_tier: None
- system_fingerprint: fp_0705bf87c0
- usage: CompletionUsage(completion_tokens=9, prompt_tokens=8, total_tokens=17, completion_tokens_details=CompletionTokensDetails(audio_tokens=0, reasoning_tokens=0, accepted_prediction_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

</details>